# NATS JetStream Stream Demo

Hands-on presentation notebook covering Stream Lifecycle (Creation, In-Place Edits & Sealing, Inspection, Sequence Handling), Storage Dynamics (File/Memory Persistence & Volatility, Retention Policies, CleanUp Discard Policies, Backup & Recovery), and Cluster Operations (Placement, Dynamic Replica Scaling, Leadership Step Down, Balancing, Peer Evacuation).

## 1. Stream Lifecycle

### Creation

Create the EVENTS stream using stream.json configuration.

In [ ]:
!nats stream add EVENTS --config=stream.json 2>&1

### In-Place Updation

Dynamically update the EVENTS stream configuration in-place using stream-update.json (adds subject inventory.*, updates metadata).

In [ ]:
!nats stream edit EVENTS --config=stream-update.json -f 2>&1

### Inspection

Publish 10 sample order events into the active stream.

In [ ]:
!nats pub --jetstream order.placed --count=10 "Order Event {{{{Count}}}}" 2>&1

Inspect current stream state, active message sequence pointers, and storage byte usage.

In [ ]:
!nats stream state EVENTS 2>&1

View stored message at sequence 1.

In [ ]:
!nats stream get EVENTS 1 2>&1

Retrieve a specific stored message directly by stream sequence number 2.

In [ ]:
!nats stream get EVENTS 2 2>&1

### Sequence Handling

Purge messages below sequence 4 using --seq=4 to advance the first sequence pointer.

In [ ]:
!nats stream purge EVENTS --seq=4 -f 2>&1

Purge stream messages while retaining only the last 3 messages using --keep=3.

In [ ]:
!nats stream purge EVENTS --keep=3 -f 2>&1

Purge all remaining messages from the stream.

In [ ]:
!nats stream purge EVENTS -f 2>&1

### Stream Sealing (Immutable Read-Only Archive)

Seal the EVENTS stream to permanently freeze it as an immutable read-only archive.

In [ ]:
!nats stream seal EVENTS -f 2>&1

Inspect stream info to verify sealed state properties (Sealed: true, DenyDelete: true, DenyPurge: true).

In [ ]:
!nats stream info EVENTS 2>&1

Attempt publishing a new message to the sealed stream (rejected by server with stream sealed error 10109).

In [ ]:
!nats pub --jetstream order.placed "Blocked Event on Sealed Stream" 2>&1

Attempt purging the sealed stream (rejected by server).

In [ ]:
!nats stream purge EVENTS -f 2>&1

> **Note**: Stream `EVENTS` remains active in the cluster as a sealed 3-replica memory stream.

## 2. Storage & Policies

### Storage Dynamics (Replicas R=2)

Create FILE_STREAM using File storage and 2 replicas for persistence testing.

In [ ]:
!nats stream add FILE_STREAM --subjects="file.events" --storage=file --replicas=2 --defaults 2>&1

Publish sample messages to File storage stream.

In [ ]:
!nats pub --jetstream file.events --count=5 "File Event {{{{Count}}}}" 2>&1

Restart cluster containers via podman-compose to test file persistence.

In [ ]:
!podman-compose -f ../../deploy/local-nats-cluster/docker-compose.yaml down 2>&1 && podman-compose -f ../../deploy/local-nats-cluster/docker-compose.yaml up -d 2>&1 && sleep 5

Inspect stream state after cluster restart to verify File storage data persistence (retains 5 messages).

In [ ]:
!nats stream info FILE_STREAM 2>&1

Create MEMORY_STREAM using Memory storage and 2 replicas for volatility testing.

In [ ]:
!nats stream add MEMORY_STREAM --subjects="memory.events" --storage=memory --replicas=2 --defaults 2>&1

Publish sample messages to Memory storage stream.

In [ ]:
!nats pub --jetstream memory.events --count=5 "Memory Event {{{{Count}}}}" 2>&1

Restart cluster containers via podman-compose to test memory volatility.

In [ ]:
!podman-compose -f ../../deploy/local-nats-cluster/docker-compose.yaml down 2>&1 && podman-compose -f ../../deploy/local-nats-cluster/docker-compose.yaml up -d 2>&1 && sleep 5

Inspect stream state after cluster restart to verify Memory storage data volatility (shows 0 messages / stream reset).

In [ ]:
!nats stream info MEMORY_STREAM 2>&1

### Retention Policies

#### 1. Limits Retention (LIMITS_STREAM - Replicas R=1)

Create LIMITS_STREAM with 1 replica, File storage, and max-msgs=5 capacity limit.

In [ ]:
!nats stream add LIMITS_STREAM --subjects="limits.events" --retention=limits --storage=file --max-msgs=5 --replicas=1 --defaults 2>&1

Publish 10 messages to exceed max-msgs=5 capacity.

In [ ]:
!nats pub --jetstream limits.events --count=10 "Limits Message {{{{Count}}}}" 2>&1

Inspect stream info to confirm limits retention (retains only last 5 messages).

In [ ]:
!nats stream info LIMITS_STREAM 2>&1

#### 2. Interest Retention (INTEREST_STREAM - Replicas R=2)

Create INTEREST_STREAM with 2 replicas.

In [ ]:
!nats stream add INTEREST_STREAM --subjects="interest.events" --retention=interest --storage=file --replicas=2 --defaults 2>&1

Register consumers C1 and C2 on INTEREST_STREAM.

In [ ]:
!nats consumer add INTEREST_STREAM C1 --filter="interest.events" --ack=explicit --pull --defaults 2>&1
!nats consumer add INTEREST_STREAM C2 --filter="interest.events" --ack=explicit --pull --defaults 2>&1

Publish a test event into INTEREST_STREAM.

In [ ]:
!nats pub --jetstream interest.events "Interest Event 1" 2>&1

Inspect stream info before consuming (message stored awaiting consumer ACKs).

In [ ]:
!nats stream info INTEREST_STREAM 2>&1

Consume and ACK with Consumer C1.

In [ ]:
!echo y | nats consumer next INTEREST_STREAM C1 --ack --timeout=3s 2>&1

Inspect stream info after C1 ACK (message retained because Consumer C2 has not ACKed yet).

In [ ]:
!nats stream info INTEREST_STREAM 2>&1

Consume and ACK with Consumer C2.

In [ ]:
!echo y | nats consumer next INTEREST_STREAM C2 --ack --timeout=3s 2>&1

Inspect stream info after C2 ACK (message automatically purged once all interested consumers ACK).

In [ ]:
!nats stream info INTEREST_STREAM 2>&1

#### 3. WorkQueue Retention (WORKQUEUE_STREAM - Replicas R=3)

Create WORKQUEUE_STREAM with 3 replicas and register WORKER consumer.

In [ ]:
!nats stream add WORKQUEUE_STREAM --subjects="workqueue.events" --retention=workq --storage=file --replicas=3 --defaults 2>&1

Register WORKER consumer on WORKQUEUE_STREAM.

In [ ]:
!nats consumer add WORKQUEUE_STREAM WORKER --filter="workqueue.events" --ack=explicit --pull --defaults 2>&1

Publish a job task into WORKQUEUE_STREAM.

In [ ]:
!nats pub --jetstream workqueue.events "Process Payment Task" 2>&1

Inspect stream info before worker processing (task stored in queue).

In [ ]:
!nats stream info WORKQUEUE_STREAM 2>&1

Consume and ACK task with WORKER consumer.

In [ ]:
!echo y | nats consumer next WORKQUEUE_STREAM WORKER --ack --timeout=3s 2>&1

Inspect stream info after worker ACK (task purged immediately upon ACK).

In [ ]:
!nats stream info WORKQUEUE_STREAM 2>&1

### CleanUp & Discard Policies

#### 1. Discard Old Policy (DISCARD_OLD_STREAM - Replicas R=1)

Create DISCARD_OLD_STREAM with 1 replica and max-msgs=5 capacity limit.

In [ ]:
!nats stream add DISCARD_OLD_STREAM --subjects="discardold.events" --retention=limits --discard=old --max-msgs=5 --storage=file --replicas=1 --defaults 2>&1

Publish 7 messages (exceeding 5-message capacity) and verify oldest messages (1 and 2) are dropped.

In [ ]:
!nats pub --jetstream discardold.events --count=7 "Message {{{{Count}}}}" 2>&1
!nats stream info DISCARD_OLD_STREAM 2>&1

#### 2. Discard New Policy (DISCARD_NEW_STREAM - Replicas R=2)

Create DISCARD_NEW_STREAM with 2 replicas and max-msgs=5 capacity limit.

In [ ]:
!nats stream add DISCARD_NEW_STREAM --subjects="discardnew.events" --retention=limits --discard=new --max-msgs=5 --storage=file --replicas=2 --defaults 2>&1

Publish 5 messages to fill capacity, then attempt publishing a 6th message (rejected by server).

In [ ]:
!nats pub --jetstream discardnew.events --count=5 "Message {{{{Count}}}}" 2>&1
!nats pub --jetstream discardnew.events "Overflow Message 6" 2>&1
!nats stream info DISCARD_NEW_STREAM 2>&1

### Backup & Recovery

Back up FILE_STREAM to local snapshot directory, simulate disaster by deleting stream, and restore from snapshot.

In [ ]:
!nats stream backup FILE_STREAM ./backup_demo 2>&1
!nats stream rm FILE_STREAM -f 2>&1
!nats stream restore ./backup_demo 2>&1
!nats stream info FILE_STREAM 2>&1

## 3. Cluster Operations

### Multi-Stream Cluster Overview

Inspect all active streams across the NATS cluster topology (varying replicas R=1, R=2, R=3).

In [ ]:
!nats stream ls 2>&1

### Replica Placement Tag Constraints

Create PLACED_STREAM with placement tag constraints (Note: requires cluster node tags in nats-server.conf).

In [ ]:
!nats stream add PLACED_STREAM --subjects="placed.*" --tags="stream:primary" --replicas=1 --defaults 2>&1

### In-Place Dynamic Replica Scaling

Dynamically scale up PLACED_STREAM replica count in-place (R=1 to R=3) live without client downtime.

In [ ]:
!nats stream edit PLACED_STREAM --replicas=3 -f 2>&1

Inspect cluster status of PLACED_STREAM after replica scaling.

In [ ]:
!nats stream info PLACED_STREAM 2>&1

### Leadership Step Down

Force active PLACED_STREAM Raft leader to step down gracefully and trigger a leader election.

In [ ]:
!nats stream cluster stepdown PLACED_STREAM 2>&1

### Cluster Balancing

Rebalance all stream replica allocations evenly across cluster nodes.

In [ ]:
!nats stream cluster balance 2>&1

### Evacuate Peer

Evacuate stream replicas away from cluster peer nats-1 (e.g. for maintenance or node retirement).

In [ ]:
!nats stream cluster evacuate nats-1 -f 2>&1

Inspect PLACED_STREAM status after peer evacuation.

In [ ]:
!nats stream info PLACED_STREAM 2>&1

### Final Cluster Teardown

Clean up all demo streams and temporary backup directory.

In [ ]:
!nats stream rm EVENTS FILE_STREAM MEMORY_STREAM LIMITS_STREAM INTEREST_STREAM WORKQUEUE_STREAM DISCARD_OLD_STREAM DISCARD_NEW_STREAM PLACED_STREAM -f 2>&1
!rm -rf ./backup_demo